In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
google_api_key=os.getenv("GOOGLE_API_KEY")

In [4]:
from llama_index.core import Document
from llama_index.core import SimpleDirectoryReader
from llama_index.core import load_index_from_storage
from llama_index.core import Settings
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.embeddings.google_genai.base import types

In [5]:
documents = SimpleDirectoryReader("data").load_data()
document = Document(text="\n\n".join([doc.text for doc in documents]))

In [6]:
def get_index(documents, index_dir):
    Settings.llm = GoogleGenAI(
    model="gemini-2.5-flash",
    api_key=google_api_key,
    generation_config=types.GenerateContentConfig(
        safety_settings=[
            types.SafetySetting(
                category= types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
                threshold=types.HarmBlockThreshold.BLOCK_NONE
            ),
            types.SafetySetting(
                category=types.HarmCategory.HARM_CATEGORY_CIVIC_INTEGRITY,
                threshold=types.HarmBlockThreshold.BLOCK_NONE
            ),
            types.SafetySetting(
                category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
                threshold=types.HarmBlockThreshold.BLOCK_NONE
            ),
            types.SafetySetting(
                category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
                threshold=types.HarmBlockThreshold.BLOCK_NONE
            ),
        ]
    ))
    Settings.embed_model = GoogleGenAIEmbedding(
        model_name="gemini-embedding-001",
        api_key=google_api_key,
        embedding_config=types.EmbedContentConfig(
            output_dimensionality=1536,
            task_type="RETRIEVAL_DOCUMENT"
        )
    )

    if not os.path.exists(index_dir):
        sentence_index = VectorStoreIndex.from_documents([document])
        sentence_index.storage_context.persist(persist_dir=index_dir)
        
    else:
        sentence_index = load_index_from_storage(StorageContext.from_defaults(persist_dir=index_dir))
    return sentence_index

In [7]:
def get_engine(sentence_index):
    engine = sentence_index.as_query_engine(similarity_top_k=6)
    return engine

In [8]:
index_dir = "basic_index_1"
bs_index_1 = get_index(documents, index_dir)
bs_engine_1 = get_engine(bs_index_1)

2025-09-20 03:30:37,462 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash "HTTP/1.1 200 OK"
2025-09-20 03:30:46,747 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
2025-09-20 03:30:48,597 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
2025-09-20 03:30:49,903 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
2025-09-20 03:30:51,502 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
2025-09-20 03:30:52,963 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
2025-09-20 03:30:54,515 - IN

In [9]:
res = bs_engine_1.query(
    "How can I calculate the final velocity if I have initial velocity and acceleration after 10 seconds?"
)
res.response

2025-09-20 03:31:11,160 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
2025-09-20 03:31:11,184 - INFO - AFC is enabled with max remote calls: 10.
2025-09-20 03:31:15,329 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2025-09-20 03:31:15,331 - INFO - AFC remote call 1 is done.


'To determine the final velocity when you have the initial velocity and acceleration after 10 seconds, you can use the kinematic equation for uniformly accelerated motion:\n\n`v = v0 + at`\n\nIn this equation:\n*   `v` represents the final velocity.\n*   `v0` represents the initial velocity.\n*   `a` represents the uniform (constant) acceleration.\n*   `t` represents the time interval, which in this case is 10 seconds.'

In [10]:
eval_questions=[
    "State Newton’s First Law of Motion. Why is it also called the Law of Inertia?",
    "A body of mass 5 kg is acted upon by two forces of 10 N and 20 N at right angles to each other. Find the resultant acceleration of the body.",
    "Explain Newton’s Third Law of Motion with two real-life examples.",
    "A truck of mass 2000 kg moves with a uniform velocity of 54 km/h. Calculate the force required to stop it in 10 seconds.",
    "A projectile is fired with an initial velocity of 20 m/s at an angle of 30° with the horizontal. Find the time of flight, maximum height, and horizontal range.",
    "A river is 500 m wide and flows at 3 m/s. A boat has a speed of 5 m/s in still water. Find the time taken by the boat to cross the river if it heads perpendicular to the bank, and also find the drift.",
    "Derive the formula for centripetal acceleration and explain why a car skids on a curved road if the speed is too high.",
    "State and prove the work-energy theorem.",
    "A spring of spring constant 200 N/m is compressed by 0.1 m. Find the potential energy stored in the spring.",
    "A body of mass 2 kg is lifted vertically to a height of 10 m. Find the work done against gravity and the potential energy gained."
]

In [11]:
from trulens.core.session import TruSession
from trulens.apps.llamaindex import TruLlama
from trulens.providers.google import Google
from trulens.core import Feedback
import numpy as np

tru = TruSession()
tru.reset_database()
# Initialize provider class
provider = Google()

def get_evaluation_response(rag_engine, app_id, eval_questions):
    # Define a groundedness feedback function
    f_groundedness = (
    Feedback(provider.groundedness_measure_with_cot_reasons,name="Groundedness")
    .on_context(collect_list=True)
    .on_output()
    )

    # Question/answer relevance
    f_answer_relevance = (
    Feedback(provider.relevance_with_cot_reasons, name="Answer Relevance")
    .on_input()
    .on_output()
    )

    # Question/context relevance
    f_context_relevance =  (
    Feedback(provider.context_relevance_with_cot_reasons,name="Context Relevance")
    .on_input()
    .on_context(collect_list=True)
    .aggregate(np.mean)
    )

    # Recorder
    tru_query_engine_recorder = TruLlama(
        rag_engine,
        app_id=app_id,
        feedbacks=[f_groundedness, f_answer_relevance, f_context_relevance]
    )

    for question in eval_questions:
        with tru_query_engine_recorder as recording:
            rag_engine.query(question)

    records = recording.get()
    return records


d:\Projects\AI Legal Research Agent\.venv\Lib\site-packages\munch\__init__.py:24: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2025-09-20 03:31:47,822 - INFO - Context impl SQLiteImpl.
2025-09-20 03:31:47,822 - INFO - Will assume non-transactional DDL.


🦑 Initialized with db url sqlite:///default.sqlite .
🛑 Secret keys may be written to the database. See the `database_redact_keys` option of `TruSession` to prevent this.


2025-09-20 03:31:48,166 - INFO - ✅ OpenTelemetry exporter set: NoneType
2025-09-20 03:31:48,221 - INFO - ✅ Added new TrulensOtelSpanProcessor
2025-09-20 03:31:48,240 - INFO - Context impl SQLiteImpl.
2025-09-20 03:31:48,241 - INFO - Will assume non-transactional DDL.


✅ experimental Feature.OTEL_TRACING enabled.
🔒 experimental Feature.OTEL_TRACING is enabled and cannot be changed.


Updating app_name and app_version in apps table: 0it [00:00, ?it/s]
Updating app_id in records table: 0it [00:00, ?it/s]
Updating app_json in apps table: 0it [00:00, ?it/s]


In [12]:
records = get_evaluation_response(
    bs_engine_1,
    app_id='basic engine',
    eval_questions = eval_questions
)


instrumenting <class 'llama_index.embeddings.google_genai.base.GoogleGenAIEmbedding'> for base <class 'llama_index.embeddings.google_genai.base.GoogleGenAIEmbedding'>
instrumenting <class 'llama_index.embeddings.google_genai.base.GoogleGenAIEmbedding'> for base <class 'llama_index.core.base.embeddings.base.BaseEmbedding'>
instrumenting <class 'llama_index.embeddings.google_genai.base.GoogleGenAIEmbedding'> for base <class 'llama_index.core.schema.TransformComponent'>
instrumenting <class 'llama_index.embeddings.google_genai.base.GoogleGenAIEmbedding'> for base <class 'llama_index.core.schema.BaseComponent'>
instrumenting <class 'llama_index.embeddings.google_genai.base.GoogleGenAIEmbedding'> for base <class 'pydantic.main.BaseModel'>
instrumenting <class 'llama_index.embeddings.google_genai.base.GoogleGenAIEmbedding'> for base <class 'llama_index_instrumentation.DispatcherSpanMixin'>
instrumenting <class 'llama_index.embeddings.google_genai.base.GoogleGenAIEmbedding'> for base <class '

d:\Projects\AI Legal Research Agent\.venv\Lib\site-packages\trulens\feedback\llm_provider.py:2134: UserWarning: Failed to process and remove trivial statements. Proceeding with all statements.
  warnings.warn(


d:\Projects\AI Legal Research Agent\.venv\Lib\site-packages\trulens\feedback\llm_provider.py:2134: UserWarning: Failed to process and remove trivial statements. Proceeding with all statements.
  warnings.warn(
d:\Projects\AI Legal Research Agent\.venv\Lib\site-packages\trulens\feedback\llm_provider.py:2134: UserWarning: Failed to process and remove trivial statements. Proceeding with all statements.
  warnings.warn(
d:\Projects\AI Legal Research Agent\.venv\Lib\site-packages\trulens\feedback\llm_provider.py:2134: UserWarning: Failed to process and remove trivial statements. Proceeding with all statements.
  warnings.warn(
d:\Projects\AI Legal Research Agent\.venv\Lib\site-packages\trulens\feedback\llm_provider.py:2134: UserWarning: Failed to process and remove trivial statements. Proceeding with all statements.
  warnings.warn(
d:\Projects\AI Legal Research Agent\.venv\Lib\site-packages\trulens\feedback\llm_provider.py:2134: UserWarning: Failed to process and remove trivial statements.

In [15]:
display(records)
tru.run_dashboard()

Starting dashboard ...
Dashboard already running at path:   Local URL: http://localhost:63459



<Popen: returncode: None args: ['streamlit', 'run', '--server.headless=True'...>

![alt text](<Screenshot 2025-09-20 032956.png>)

In [16]:
tru.stop_dashboard()

C:\Users\prata\AppData\Local\Temp\ipykernel_11180\2300421333.py:1: DeprecationWarning: Method `stop_dashboard` has been renamed or moved to `trulens.dashboard.run.stop_dashboard`.

  tru.stop_dashboard()
